# Project Bài 4 · Dự báo số ngày giao hàng bằng Gradient Descent tự cài đặt

**Học máy cơ bản · Bài 4 · PGS.TS Nguyễn Văn Hậu**

Mục tiêu: xây một mô hình hồi quy tuyến tính dự đoán **số ngày từ lúc xác nhận đơn đến lúc giao**, tự cài đặt Gradient Descent (GD), đánh giá đúng quy trình và viết kết luận có phạm vi.

| Mốc | Bước | Nộp | Ô tự kiểm |
|---|---|---|---|
| M1 | 1–2: định nghĩa bài toán, mốc đối chứng | sau buổi học | đặc trưng hợp lệ, MAE mốc trên validation |
| M2 | 3: GD hai tham số với khoảng cách | tuần sau | một bước GD đúng như slide; GD khớp nghiệm bình phương tối thiểu |
| M3 | 4–6: GD dạng vector, chọn trên validation, chấm test một lần, báo cáo | cuối đợt | GD vector khớp nghiệm; `cham_test` được gọi đúng một lần |

**Mức cơ bản** (tối đa 8/10): làm bước 1–3, 5 và 6 với một đặc trưng; đặt `LAM_BUOC_4 = False` ở bước 4.

**Cách dùng.** Chạy từ trên xuống. Ô có `TODO` là phần bạn tự viết. Ô bắt đầu bằng `# TỰ KIỂM` không được sửa: nó in ✅ khi đúng, hoặc báo lỗi kèm gợi ý. Ô tự kiểm chỉ kiểm phép tính; phần diễn giải được chấm theo rubric.

**Quy tắc quy trình.** (1) Mọi phép biến đổi (điền giá trị thiếu, chuẩn hoá) chỉ fit trên **train**. (2) Chọn learning rate và mô hình bằng **train/validation**. (3) **Test chỉ chấm một lần**, sau khi đã chốt lựa chọn.

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'prepared' / 'olist_model_table.csv'
assert DATA.exists(), 'Mở notebook từ thư mục bài (có thư mục data/prepared).'
d = pd.read_csv(DATA)
np.random.seed(2026)

def mae(y, p):  return float(np.mean(np.abs(np.asarray(p) - np.asarray(y))))
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

print(d.shape)
d.head(3)

## Bước 1 · Định nghĩa bài toán (Chặng 1)

- Đơn vị quan sát: **một đơn hàng** (một dòng).
- Nhãn: `delivery_days` = số ngày từ xác nhận đơn đến giao thực tế.
- Thời điểm dự đoán: **lúc xác nhận đơn** (`order_approved_at`).

Bảng dưới liệt kê các cột. Một cột chỉ được làm đầu vào nếu **đã có tại lúc xác nhận đơn**. Theo giả định của case, giá, phí vận chuyển, khối lượng, số mặt hàng, khoảng cách và ngày giao dự kiến (`promised_days`) có sẵn lúc xác nhận. Ngày giao thực tế và nhãn trễ chỉ có sau khi giao.

| Cột | Ý nghĩa |
|---|---|
| `order_id` | mã đơn (định danh) |
| `order_approved_at` | thời điểm xác nhận (dùng để chia tập) |
| `order_delivered_customer_date` | ngày giao thực tế |
| `order_estimated_delivery_date` | ngày giao dự kiến |
| `distance_km` | khoảng cách gần đúng người bán–người mua |
| `promised_days` | số ngày đã hứa (từ ngày dự kiến) |
| `price`, `freight_value` | giá, phí vận chuyển (BRL) |
| `n_items`, `weight_g` | số mặt hàng, tổng khối lượng (g) |
| `delivery_days` | **nhãn** |
| `late` | giao trễ (tạo từ ngày giao thực tế) |
| `split` | train / validation / test (đã chia theo thời gian) |

**Việc của bạn:** chọn danh sách `FEATURES` (ít nhất 3 cột, phải có `distance_km`).

In [ ]:
# TODO: liệt kê ít nhất 3 đặc trưng có sẵn lúc xác nhận đơn, phải có 'distance_km'
FEATURES = [...]
TARGET = 'delivery_days'
train, val, test = (d[d['split'] == s].copy() for s in ['train', 'validation', 'test'])
y_tr, y_va, y_te = (t[TARGET].to_numpy() for t in (train, val, test))
print('Số đơn train / validation / test:', len(train), len(val), len(test))

In [ ]:
# TỰ KIỂM · Bước 1
TUONG_LAI = {'order_delivered_customer_date', 'late', 'delivery_days', 'split', 'order_id', 'order_approved_at', 'order_estimated_delivery_date'}
kiem('Không dùng cột tương lai hoặc cột định danh', not (set(FEATURES) & TUONG_LAI),
     'Ngày giao thực tế và nhãn trễ chỉ có sau khi giao; mã đơn và thời điểm không phải đặc trưng số.')
kiem('Có khoảng cách', 'distance_km' in FEATURES)
kiem('Ít nhất 3 đặc trưng, đều có trong bảng', len(FEATURES) >= 3 and set(FEATURES) <= set(d.columns))

## Bước 2 · Mốc đối chứng (Chặng 4)

Mốc là dự đoán **hằng** bằng trung bình `y` của **train**. Tính MAE và RMSE của mốc trên **validation**.

In [ ]:
# TODO: moc = trung bình y của TRAIN; dự đoán hằng cho validation; tính MAE, RMSE
moc = ...
moc_mae_va, moc_rmse_va = ..., ...
print(f'Mốc: {moc:.3f} ngày · validation MAE {moc_mae_va:.3f} · RMSE {moc_rmse_va:.3f}')

In [ ]:
# TỰ KIỂM · Bước 2 (mốc M1)
kiem('Mốc dùng trung bình train', np.isclose(moc, y_tr.mean()), 'Không dùng y của validation hoặc test.')
kiem('MAE mốc trên validation ≈ 5,364 ngày', np.isclose(moc_mae_va, 5.3644, atol=5e-4), 'Dự đoán hằng = trung bình train cho MỌI đơn validation.')

## Bước 3 · GD hai tham số với khoảng cách (Chặng 2–3)

1. Điền giá trị thiếu của `distance_km` bằng **trung vị train**, rồi chuẩn hoá z-score bằng **trung bình và độ lệch chuẩn train**.
2. Viết `gd_2(x, y, eta, steps, w0, b0)`: mỗi bước tính **cả hai gradient tại cùng (w, b)** rồi mới cập nhật. Lưu `J = MSE/2` trước mỗi bước.

In [ ]:
# TODO: trung vị, trung bình, độ lệch chuẩn đều tính trên TRAIN; dùng lại cho validation và test
med = ...
km_tr, km_va, km_te = ...
mu, sd = ...
x_tr, x_va, x_te = ...

In [ ]:
# TỰ KIỂM · chuẩn hoá theo train
kiem('Trung vị lấy từ train', np.isclose(med, train['distance_km'].median()))
kiem('Train sau chuẩn hoá có trung bình 0, độ lệch chuẩn 1', abs(x_tr.mean()) < 1e-9 and abs(x_tr.std() - 1) < 1e-9)
kiem('Validation dùng cùng mu, sd của train', np.allclose(x_va * sd + mu, km_va), 'Không fit lại trên validation.')

In [ ]:
def gd_2(x, y, eta, steps, w0=0.0, b0=0.0):
    w, b = w0, b0
    history = []
    for _ in range(steps):
        # TODO: sai số e; lưu J = MSE/2; tính g_w, g_b tại cùng (w, b); rồi cập nhật cả hai
        ...
    return w, b, np.array(history)

In [ ]:
# TỰ KIỂM · một bước GD đúng như slide (4 điểm mô phỏng, eta = 0,05, từ (0,5; 1))
xt, yt = np.array([1., 2., 3., 4.]), np.array([3., 4., 6., 7.])
w1, b1, h1 = gd_2(xt, yt, eta=0.05, steps=1, w0=0.5, b0=1.0)
kiem('Bước hai tham số: (0,5; 1) → (0,9; 1,1375)', np.allclose([w1, b1], [0.9, 1.1375]),
     'Tính g_w và g_b từ cùng (w, b) cũ; gradient đúng là (−8; −2,75).')
kiem('J trước bước = 4,3125', np.isclose(h1[0], 4.3125), 'J = MSE/2.')

Thử ba learning rate trên **train** và chọn một giá trị cho loss giảm ổn định. Không dùng validation hay test để chọn `eta` ở bước này.

In [ ]:
ket_qua = {}
for eta in [0.01, 0.1, 0.5]:
    ket_qua[eta] = gd_2(x_tr, y_tr, eta=eta, steps=300)
    plt.plot(ket_qua[eta][2], label=f'eta = {eta}')
plt.yscale('log'); plt.xlabel('Bước'); plt.ylabel('J trên train (ngày²)'); plt.legend(); plt.show()
ETA = ...   # TODO: chọn từ đồ thị loss trên train
w_gd, b_gd, _ = ket_qua[ETA]

In [ ]:
# TỰ KIỂM · GD khớp nghiệm bình phương tối thiểu (mốc M2)
A = np.column_stack([x_tr, np.ones_like(x_tr)])
w_ols, b_ols = np.linalg.lstsq(A, y_tr, rcond=None)[0]
kiem('GD khớp nghiệm: sai khác < 0,001', abs(w_gd - w_ols) < 1e-3 and abs(b_gd - b_ols) < 1e-3,
     'Tăng số bước hoặc chọn eta khác; kiểm lại công thức gradient.')
print(f'Hệ chuẩn hoá: w = {w_gd:.4f}, b = {b_gd:.4f}')

In [ ]:
# Đổi về đơn vị km để diễn giải
w_km = w_gd / sd
b_km = b_gd - w_gd * mu / sd
print(f'ŷ ≈ {b_km:.3f} + {w_km:.5f}·km   (mỗi 100 km thêm khoảng {100 * w_km:.2f} ngày)')
print(f'Dự đoán cho 600 km: {b_km + w_km * 600:.2f} ngày')
# Thử nghiệm tuỳ chọn: chạy gd_2 trên km gốc (chưa chuẩn hoá) với eta = 0,1 và ghi lại điều xảy ra.

## Bước 4 · GD dạng vector cho nhiều đặc trưng (Chặng 3)

Ma trận thiết kế `X` gồm các đặc trưng đã điền thiếu và chuẩn hoá **theo train**, thêm một cột 1 cho hệ số chặn. Gradient của `J = MSE/2` là `Xᵀ(Xθ − y)/n`.

In [ ]:
LAM_BUOC_4 = True   # Mức cơ bản: đặt False để bỏ qua bước 4; notebook vẫn chạy được đến cuối
if LAM_BUOC_4:
    # TODO: trung vị, trung bình, độ lệch chuẩn của từng cột tính trên TRAIN
    med_f = ...
    F_tr, F_va, F_te = ...
    mu_f, sd_f = ...
    def thiet_ke(F):
        ...   # TODO: chuẩn hoá từng cột rồi thêm cột 1 ở cuối
    X_tr, X_va, X_te = thiet_ke(F_tr), thiet_ke(F_va), thiet_ke(F_te)

    def gd_vec(X, y, eta, steps, theta0=None):
        theta = np.zeros(X.shape[1]) if theta0 is None else np.array(theta0, float)
        history = []
        for _ in range(steps):
            ...   # TODO: e = Xθ − y; lưu J; θ ← θ − eta · Xᵀe / n
        return theta, np.array(history)

In [ ]:
# TỰ KIỂM · GD vector
if LAM_BUOC_4:
    Xt = np.column_stack([xt, np.ones(4)])
    th1, _ = gd_vec(Xt, yt, eta=0.05, steps=1, theta0=[0.5, 1.0])
    kiem('Một bước vector trùng với bước hai tham số', np.allclose(th1, [0.9, 1.1375]), 'Gradient = Xᵀe / n.')
    theta, hist = gd_vec(X_tr, y_tr, eta=0.1, steps=2000)
    theta_ols = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]
    kiem('GD vector khớp nghiệm: sai khác < 0,001', np.max(np.abs(theta - theta_ols)) < 1e-3,
         'Kiểm lại chuẩn hoá theo train và công thức gradient; thử tăng số bước.')
    plt.plot(hist); plt.yscale('log'); plt.xlabel('Bước'); plt.ylabel('J trên train'); plt.show()
else:
    print('⏭ Bỏ qua bước 4 (mức cơ bản)')

## Bước 5 · Chọn trên validation, chấm test một lần (Chặng 4)

In [ ]:
ung_vien = {
    'Mốc trung bình': (lambda: np.full(len(y_va), moc)),
    'GD · khoảng cách': (lambda: b_gd + w_gd * x_va),
}
if LAM_BUOC_4:
    ung_vien[f'GD · {len(FEATURES)} đặc trưng'] = lambda: X_va @ theta
bang = pd.DataFrame([{'mô hình': k, 'MAE val': mae(y_va, f()), 'RMSE val': rmse(y_va, f())} for k, f in ung_vien.items()])
print(bang.round(3).to_string(index=False))
CHON = ...   # TODO: tên mô hình có MAE validation nhỏ nhất
print('Chọn:', CHON)

In [ ]:
# TỰ KIỂM · chọn bằng validation
kiem('CHON là mô hình có MAE validation nhỏ nhất trong bảng', CHON == bang.loc[bang['MAE val'].idxmin(), 'mô hình'],
     'Bảng phải tính trên validation; không nhìn test khi chọn.')

In [ ]:
# Chấm test MỘT LẦN, cùng lúc cho mô hình đã chọn và mốc (mốc chỉ để đối chiếu, không dùng để chọn).
_SO_LAN_CHAM = globals().get('_SO_LAN_CHAM', 0)
def cham_test(du_doan, du_doan_moc):
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã được chấm. Muốn đổi mô hình: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    return mae(y_te, du_doan), rmse(y_te, du_doan), mae(y_te, du_doan_moc), rmse(y_te, du_doan_moc)

du_doan_test = {'Mốc trung bình': lambda: np.full(len(y_te), moc), 'GD · khoảng cách': lambda: b_gd + w_gd * x_te,
                f'GD · {len(FEATURES)} đặc trưng': lambda: X_te @ theta}[CHON]()
test_mae, test_rmse, moc_mae_te, moc_rmse_te = cham_test(du_doan_test, np.full(len(y_te), moc))
print(f'{CHON}: MAE test = {test_mae:.3f} ngày · RMSE test = {test_rmse:.3f} ngày')
print(f'Mốc trên test (chỉ để đối chiếu): MAE {moc_mae_te:.3f} · RMSE {moc_rmse_te:.3f}')
sai_so = du_doan_test - y_te
plt.hist(np.clip(sai_so, -40, 40), bins=np.linspace(-40, 40, 41)); plt.axvline(0, c='orange')
plt.xlabel('Dự đoán − thực tế (ngày)'); plt.ylabel('Số đơn'); plt.show()

In [ ]:
# TỰ KIỂM · quy trình (mốc M3)
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Chạy lại toàn bộ notebook (Restart & Run All) nếu đã gọi nhiều lần.')

## Bước 6 · Báo cáo một trang

Sao chép khung dưới vào một ô Markdown mới hoặc tệp báo cáo; mỗi mục 2–4 câu.

1. **Bài toán và dữ liệu.** Dự đoán gì, tại thời điểm nào, trên nhóm đơn nào; số đơn mỗi tập.
2. **Phương pháp.** Đặc trưng đã chọn và lý do; chuẩn hoá theo train; GD (eta, số bước) và cách đối chiếu với nghiệm bình phương tối thiểu.
3. **Kết quả.** Bảng validation; mô hình được chọn; MAE và RMSE trên test kèm mốc; một hình (phân phối sai số).
4. **Kết luận có phạm vi.** Mô hình tốt hơn mốc bao nhiêu ngày, trên nhóm đơn nào; kết luận chưa áp dụng được cho đâu.
5. **Ba giới hạn.** Ví dụ: chỉ đơn đã giao; khoảng cách gần đúng; giả định thời điểm sẵn có của cột; quan hệ tuyến tính; không phải nhân quả.
6. **Việc làm tiếp.** Một thay đổi bạn sẽ thử và cách kiểm nó **không** dùng test.

## Nâng cao (tuỳ chọn)

- **Mini-batch GD:** viết `gd_minibatch(X, y, eta, so_epoch, co_lo)` lấy ngẫu nhiên các lô (seed cố định). So sánh J trên train theo số lượt đọc dữ liệu với GD toàn bộ train.
- **Phân tích sai số theo khoảng cách:** chia validation thành 4 nhóm khoảng cách (theo tứ phân vị của train), tính MAE từng nhóm. Nhóm nào khó nhất? Đây là mô tả, chưa phải nguyên nhân.